# Day 12 — Document Chunking Experiment

### Tools: Python + LangChain

## Project goal

Large documents are usually too long to send to a model or search as one giant block. **Chunking** divides a document into smaller pieces so each piece can be retrieved or processed independently.

This project compares two approaches:

1. **Manual fixed-size chunking** written from scratch with a Python loop.
2. **LangChain `RecursiveCharacterTextSplitter`**, which tries to preserve larger natural text units before falling back to smaller separators.

We will experiment with:

- chunk sizes: **100, 300, 500, 1000 characters**
- overlaps: **0, 50, 100 characters**
- total configurations: **12**

For the same retrieval query, we will compare the chunks returned by every configuration, inspect boundary failures, and recommend a chunking strategy.

> **Important:** The notebook fetches the current Wikipedia article at runtime instead of copying a long copyrighted article into the notebook. The fetched content becomes a normal Python `str` variable called `long_text`.

## 1. Why do we chunk documents?

Imagine a document contains 30,000 characters.

If we search the whole document as one unit, we have two problems:

- the result may contain far more text than we need;
- the useful passage may be buried inside a very large document.

Chunking changes this:

```text
One long document
        ↓
┌────────┬────────┬────────┬────────┐
│ chunk 1│ chunk 2│ chunk 3│ chunk 4│ ...
└────────┴────────┴────────┴────────┘
```

Now retrieval can return the **small section containing the answer** instead of the entire document.

### The difficult part

If we split at the wrong place, we can separate one thought:

```text
Chunk A: "The system works because the model first..."
Chunk B: "learns patterns from the training data."
```

Chunk B has lost part of the context.

That is why this project studies both **chunk size** and **overlap**.

# 2. Install/import the required tools

Current LangChain documentation uses the `langchain-text-splitters` package for `RecursiveCharacterTextSplitter`. The official documentation shows:

```python
from langchain_text_splitters import RecursiveCharacterTextSplitter
```

and describes it as a general-purpose splitter that tries larger separators such as paragraphs and newlines before falling back to spaces and individual characters. citeturn1search0turn1search1

In [ ]:
import sys
import subprocess

# Install the current LangChain text-splitting package if needed.
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-qU",
    "langchain-text-splitters"
])


### New command: `%pip`

`%pip install ...` is a Jupyter/IPython command.

- `pip` installs Python packages.
- `-q` means quiet output.
- `-U` means upgrade to the latest compatible version.

We use `langchain-text-splitters` because current LangChain documentation places the text splitter there. citeturn1search0

In [ ]:
import re
import math
import urllib.parse
import urllib.request
from html.parser import HTMLParser

from langchain_text_splitters import RecursiveCharacterTextSplitter

### New imports explained

- `re` — Python's regular-expression module. We use it to clean extracted HTML text.
- `math` — provides mathematical functions such as `sqrt`.
- `urllib.parse` — safely puts an article title into a URL.
- `urllib.request` — Python's built-in way to make an HTTP request.
- `HTMLParser` — a Python standard-library class for reading HTML.
- `RecursiveCharacterTextSplitter` — LangChain's recursive text splitter.

No separate web-scraping library is required.

# 3. Load a long Wikipedia article as a plain Python string

We will use the **Artificial intelligence** Wikipedia article.

The MediaWiki REST API provides a page endpoint, and MediaWiki's documentation shows Python access to Wikipedia REST endpoints. citeturn0search0turn0search1

We fetch the article's HTML and convert the visible page text into a plain Python string.

In [ ]:
ARTICLE_TITLE = "Artificial intelligence"

encoded_title = urllib.parse.quote(ARTICLE_TITLE.replace(" ", "_"))
url = f"https://en.wikipedia.org/w/rest.php/v1/page/{encoded_title}/html"

request = urllib.request.Request(
    url,
    headers={
        "User-Agent": "NLP-Chunking-Educational-Notebook/1.0"
    }
)

with urllib.request.urlopen(request, timeout=30) as response:
    html = response.read().decode("utf-8")

print("Downloaded HTML characters:", len(html))

### Understanding the request

```python
urllib.parse.quote(...)
```

converts a title into a URL-safe form.

```python
urllib.request.Request(...)
```

creates an HTTP request.

The `User-Agent` identifies our educational notebook to the server.

```python
urlopen(...)
```

actually sends the request and receives the page.

We use the `/html` endpoint because the current MediaWiki REST API documents that route for obtaining page HTML. citeturn0search1

In [ ]:
class VisibleTextExtractor(HTMLParser):
    """Collect visible text from HTML while ignoring scripts/styles."""

    def __init__(self):
        super().__init__()
        self.parts = []
        self.ignore_depth = 0

    def handle_starttag(self, tag, attrs):
        if tag.lower() in {"script", "style", "noscript"}:
            self.ignore_depth += 1

    def handle_endtag(self, tag):
        if tag.lower() in {"script", "style", "noscript"} and self.ignore_depth:
            self.ignore_depth -= 1

    def handle_data(self, data):
        if self.ignore_depth == 0:
            cleaned = re.sub(r"\s+", " ", data).strip()
            if cleaned:
                self.parts.append(cleaned)


parser = VisibleTextExtractor()
parser.feed(html)

long_text = "\n\n".join(parser.parts)

print("Article title:", ARTICLE_TITLE)
print("Plain-text characters:", len(long_text))
print("Approximate pages:", round(len(long_text) / 3000, 1))

### Why create `long_text`?

The assignment asks for the document as a **plain string variable in Python**.

That is exactly what this line creates:

```python
long_text = "\n\n".join(parser.parts)
```

The article is now represented in memory as:

```python
type(long_text)
```

which should be:

```text
str
```

We use approximately **3,000 characters per page** as a rough educational estimate. The exact number of pages depends on font, margins, spacing, and page layout.

The check below makes sure the downloaded document is sufficiently long for the experiment.

In [ ]:
MINIMUM_CHARACTERS = 15000

if len(long_text) < MINIMUM_CHARACTERS:
    raise ValueError(
        f"The downloaded article is only {len(long_text):,} characters. "
        f"Please choose a longer article."
    )

print("Requirement check passed.")
print(f"{len(long_text):,} characters is enough for a five-page-scale experiment.")
print("\nFirst 1000 characters:\n")
print(long_text[:1000])

### Why `long_text[:1000]`?

Python slicing uses:

```python
string[start:end]
```

So:

```python
long_text[:1000]
```

means:

> start at position 0 and stop before position 1000.

We only print a small preview so the notebook does not become enormous.

# 4. Manual fixed-size chunking — from scratch

Before using LangChain, we implement the mechanism ourselves.

The requirement is to split every `N` characters using a loop.

The basic idea is:

```text
start = 0
       ↓
text[start:start + chunk_size]
       ↓
move start forward
       ↓
repeat until the text ends
```

This teaches what a text splitter is actually doing underneath.

In [ ]:
def manual_fixed_chunks(text, chunk_size, chunk_overlap=0):
    """Split text into fixed-size character chunks using a Python loop."""

    if not isinstance(text, str):
        raise TypeError("text must be a string.")

    if chunk_size <= 0:
        raise ValueError("chunk_size must be greater than 0.")

    if chunk_overlap < 0:
        raise ValueError("chunk_overlap cannot be negative.")

    if chunk_overlap >= chunk_size:
        raise ValueError(
            "chunk_overlap must be smaller than chunk_size."
        )

    chunks = []
    start = 0

    step = chunk_size - chunk_overlap

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += step

    return chunks

## Understand every important line

### `chunks = []`

Creates an empty Python list where the pieces will be stored.

### `start = 0`

The first chunk begins at character position 0.

### `step = chunk_size - chunk_overlap`

This is important.

Suppose:

```text
chunk_size = 100
chunk_overlap = 20
```

Then:

```text
step = 100 - 20
     = 80
```

The next chunk starts 80 characters after the previous start, so 20 characters are shared.

### `while start < len(text):`

Keep creating chunks until the starting position reaches the end of the document.

### `text[start:end]`

This is Python slicing.

### `start += step`

Move the window forward.

This is the core mechanics of fixed-size chunking.

In [ ]:
manual_chunks_300 = manual_fixed_chunks(
    long_text,
    chunk_size=300,
    chunk_overlap=0
)

print("Number of manual chunks:", len(manual_chunks_300))
print("\nFirst chunk:\n")
print(manual_chunks_300[0])

# 5. Check manual chunk boundaries

A useful debugging technique is to store the original character positions.

The function below creates records containing:

- chunk text
- start position
- end position

This will later help us identify exactly where sentences were split.

In [ ]:
def manual_chunks_with_positions(text, chunk_size, chunk_overlap=0):
    """Return fixed chunks together with their original character positions."""

    if chunk_size <= 0:
        raise ValueError("chunk_size must be greater than 0.")

    if not 0 <= chunk_overlap < chunk_size:
        raise ValueError(
            "chunk_overlap must satisfy 0 <= overlap < chunk_size."
        )

    chunks = []
    start = 0
    step = chunk_size - chunk_overlap

    while start < len(text):
        end = min(start + chunk_size, len(text))

        chunks.append({
            "text": text[start:end],
            "start": start,
            "end": end
        })

        start += step

    return chunks


example_chunks = manual_chunks_with_positions(
    long_text,
    chunk_size=300,
    chunk_overlap=50
)

print("First chunk positions:")
print(example_chunks[0]["start"], "to", example_chunks[0]["end"])
print("Length:", len(example_chunks[0]["text"]))

### Why use `min()`?

The final chunk may end before the requested size.

For example, if only 80 characters remain and the chunk size is 300:

```python
min(300, 80)
```

returns:

```text
80
```

so we do not pretend the chunk extends beyond the document.

# 6. LangChain recursive character splitting

Now we use the library version.

Current LangChain documentation recommends `RecursiveCharacterTextSplitter` for generic text. It tries separators in order — by default paragraph breaks, newlines, spaces, and finally individual characters — so it tries to preserve larger meaningful units before making smaller cuts. citeturn1search0turn1search1

Basic usage:

```python
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)
```

Then:

```python
chunks = splitter.split_text(long_text)
```

returns ordinary Python strings.

In [ ]:
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    length_function=len,
    is_separator_regex=False
)

recursive_chunks = recursive_splitter.split_text(long_text)

print("Number of recursive chunks:", len(recursive_chunks))
print("\nFirst recursive chunk:\n")
print(recursive_chunks[0])

## What do the parameters mean?

### `chunk_size=300`

The target maximum size is 300 characters.

### `chunk_overlap=50`

Adjacent chunks try to share about 50 characters.

Overlap helps reduce information loss when an important idea sits close to a boundary. LangChain's documentation specifically describes overlap as a way to mitigate context loss. citeturn1search0

### `length_function=len`

Chunk size is measured using Python's `len()` function, so we are measuring **characters**.

### `is_separator_regex=False`

Our separators are ordinary strings rather than regular expressions.

# 7. Build one retrieval method for every chunk configuration

The assignment asks us to run the **same retrieval query** against all 12 configurations.

We will use a small, transparent lexical retrieval method built only with Python's standard library.

It works like this:

1. tokenize the query and each chunk;
2. count word frequencies;
3. calculate cosine similarity;
4. rank the chunks.

This keeps the experiment focused on **chunking**, rather than introducing another external retrieval library.

In [ ]:
def tokenize(text):
    """Convert text into lowercase alphabetic word tokens."""
    return re.findall(r"[a-zA-Z]+", text.lower())


def word_vector(text):
    """Return a word-frequency dictionary for one text."""
    counts = {}

    for word in tokenize(text):
        counts[word] = counts.get(word, 0) + 1

    return counts


def cosine_text_similarity(text_a, text_b):
    """Calculate cosine similarity between two word-frequency vectors."""

    vector_a = word_vector(text_a)
    vector_b = word_vector(text_b)

    common_words = set(vector_a) & set(vector_b)

    dot_product = sum(
        vector_a[word] * vector_b[word]
        for word in common_words
    )

    magnitude_a = math.sqrt(
        sum(value ** 2 for value in vector_a.values())
    )

    magnitude_b = math.sqrt(
        sum(value ** 2 for value in vector_b.values())
    )

    if magnitude_a == 0 or magnitude_b == 0:
        return 0.0

    return dot_product / (magnitude_a * magnitude_b)


def retrieve_chunks(query, chunks, top_k=3):
    """Return top-k chunks ranked by lexical cosine similarity."""

    scored = []

    for index, chunk in enumerate(chunks):
        score = cosine_text_similarity(query, chunk)

        scored.append({
            "index": index,
            "score": score,
            "text": chunk
        })

    scored.sort(key=lambda item: item["score"], reverse=True)

    return scored[:top_k]

## New Python ideas

### `set(vector_a) & set(vector_b)`

The `&` operator finds the intersection of two sets.

So it gives us the words appearing in **both** texts.

### `sum(...)`

Adds values together.

### `** 2`

Squares a number.

### `math.sqrt(...)`

Calculates a square root.

### `scored.sort(..., reverse=True)`

Sorts the results from highest score to lowest score.

This gives us a simple ranking system without requiring another NLP package.

# 8. Choose the retrieval query

We want a query that should occur in a meaningful section of the article.

We will search for:

> **How are artificial intelligence systems trained using data and machine learning?**

The exact wording may not appear in the article, but important terms such as `artificial intelligence`, `systems`, `trained`, `data`, and `machine learning` should occur in relevant sections.

In [ ]:
QUERY = (
    "How are artificial intelligence systems trained "
    "using data and machine learning?"
)

print("Retrieval query:")
print(QUERY)

# 9. Define all 12 configurations

There are:

```text
4 chunk sizes × 3 overlap values = 12 configurations
```

The requested values are:

- chunk sizes: 100, 300, 500, 1000
- overlaps: 0, 50, 100

In [ ]:
CHUNK_SIZES = [100, 300, 500, 1000]
OVERLAPS = [0, 50, 100]

configurations = [
    (size, overlap)
    for size in CHUNK_SIZES
    for overlap in OVERLAPS
]

print("Number of configurations:", len(configurations))

for size, overlap in configurations:
    print(f"chunk_size={size:4d}, overlap={overlap:3d}")

### New syntax: list comprehension

This:

```python
[
    (size, overlap)
    for size in CHUNK_SIZES
    for overlap in OVERLAPS
]
```

is a compact way to generate every combination.

It produces:

```text
(100, 0)
(100, 50)
(100, 100)
...
(1000, 100)
```

Notice that `overlap=100` is valid for chunk sizes 300, 500, and 1000, but **not** for chunk size 100 because overlap must be smaller than chunk size.

We therefore handle that configuration specially in the experiment.

# 10. Run the full 12-configuration experiment

For the LangChain splitter, `chunk_size` and `chunk_overlap` have the same conceptual meanings as described in the official documentation. citeturn1search0

We record:

- number of chunks
- top chunk index
- top similarity score
- top retrieved text
- average chunk length

In [ ]:
experiment_results = []

for chunk_size, overlap in configurations:

    # Overlap equal to chunk size would make the sliding window invalid.
    if overlap >= chunk_size:
        experiment_results.append({
            "chunk_size": chunk_size,
            "overlap": overlap,
            "valid": False,
            "num_chunks": None,
            "top_score": None,
            "top_chunk": None,
            "top_text": "INVALID: overlap must be smaller than chunk size."
        })
        continue

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        length_function=len,
        is_separator_regex=False
    )

    chunks = splitter.split_text(long_text)
    ranked = retrieve_chunks(QUERY, chunks, top_k=1)

    top = ranked[0]

    experiment_results.append({
        "chunk_size": chunk_size,
        "overlap": overlap,
        "valid": True,
        "num_chunks": len(chunks),
        "top_score": top["score"],
        "top_chunk": top["index"],
        "top_text": top["text"]
    })


for result in experiment_results:
    print("=" * 90)

    if not result["valid"]:
        print(
            f"chunk_size={result['chunk_size']}, "
            f"overlap={result['overlap']} → INVALID"
        )
        print(result["top_text"])
        continue

    print(
        f"chunk_size={result['chunk_size']}, "
        f"overlap={result['overlap']}"
    )
    print("Number of chunks:", result["num_chunks"])
    print("Top chunk index:", result["top_chunk"])
    print("Top similarity:", round(result["top_score"], 4))
    print("Top result:")
    print(result["top_text"])

# 11. Compare the 12 configurations

A high similarity score alone is not enough.

The assignment asks:

> Which configuration returns the most **contextually complete** result?

We therefore use two ideas:

### Retrieval score

How much lexical overlap exists between the query and the chunk?

### Contextual completeness

Does the retrieved chunk contain enough surrounding text to make the idea understandable?

A tiny chunk can have a high score because it contains the query keywords, while still cutting the explanation in half.

A larger chunk may contain slightly less keyword density but preserve the complete thought.

In [ ]:
def completeness_score(chunk):
    """Simple heuristic for contextual completeness.

    We reward:
    - multiple sentences
    - reasonable length
    - a complete-looking final sentence
    """

    sentence_count = len(
        re.findall(r"[.!?]", chunk)
    )

    score = 0

    if sentence_count >= 2:
        score += 2
    elif sentence_count == 1:
        score += 1

    if len(chunk) >= 250:
        score += 1

    stripped = chunk.strip()

    if stripped.endswith((".", "!", "?")):
        score += 2

    return score


for result in experiment_results:
    if result["valid"]:
        result["completeness"] = completeness_score(
            result["top_text"]
        )
    else:
        result["completeness"] = -1


valid_results = [
    result for result in experiment_results
    if result["valid"]
]

best_contextual = max(
    valid_results,
    key=lambda result: (
        result["completeness"],
        result["top_score"]
    )
)

print("Configuration with strongest contextual-completeness heuristic:")
print(
    f"chunk_size={best_contextual['chunk_size']}, "
    f"overlap={best_contextual['overlap']}"
)
print("Completeness score:", best_contextual["completeness"])
print("Similarity:", round(best_contextual["top_score"], 4))
print("\nRetrieved chunk:\n")
print(best_contextual["top_text"])

### Important interpretation

The `completeness_score()` is an **evaluation heuristic**, not a universal truth.

Real retrieval evaluation would use human-labelled relevance judgments or a benchmark dataset.

We use this transparent heuristic because the assignment asks us to compare contextual completeness, and it lets us explain exactly why a result was preferred.

# 12. Print a compact experiment table

This makes the 12 configurations easy to compare.

In [ ]:
print(
    f"{'Size':>6} {'Overlap':>8} {'Chunks':>8} "
    f"{'Similarity':>12} {'Complete':>10}"
)
print("-" * 52)

for result in valid_results:
    print(
        f"{result['chunk_size']:>6} "
        f"{result['overlap']:>8} "
        f"{result['num_chunks']:>8} "
        f"{result['top_score']:>12.4f} "
        f"{result['completeness']:>10}"
    )

### What should we observe?

Generally:

- **100-character chunks** are very precise but may lose context.
- **300-character chunks** often provide a useful balance.
- **500-character chunks** can preserve more complete explanations.
- **1000-character chunks** provide substantial context but may contain unrelated material.
- **Overlap** can help when an important sentence crosses a boundary.
- Too much overlap increases redundancy and the number of chunks.

There is no universally best setting. The best choice depends on document structure and retrieval goals.

# 13. Find three sentence-boundary failures

Now we address one of the most important requirements.

We need three examples where a sentence is split across a chunk boundary and report the **exact character position**.

We deliberately inspect manual fixed-size chunks because manual splitting creates easy-to-understand character boundaries.

In [ ]:
def find_sentence_boundary_failures(text, chunk_size=300, max_examples=3):
    """Find sentences whose text crosses a fixed-size chunk boundary."""

    failures = []

    # Sentence spans using a simple punctuation-based heuristic.
    sentence_pattern = re.compile(r"[^.!?]+[.!?]")

    for match in sentence_pattern.finditer(text):
        start = match.start()
        end = match.end()
        sentence = match.group().strip()

        if not sentence:
            continue

        boundary = chunk_size

        while boundary < len(text):
            if start < boundary < end:
                failures.append({
                    "sentence_start": start,
                    "sentence_end": end,
                    "boundary": boundary,
                    "sentence": sentence
                })
                break

            boundary += chunk_size

        if len(failures) >= max_examples:
            break

    return failures


boundary_failures = find_sentence_boundary_failures(
    long_text,
    chunk_size=300,
    max_examples=3
)

print("Found", len(boundary_failures), "boundary failures.")

for number, failure in enumerate(boundary_failures, start=1):
    print("=" * 90)
    print(f"Example {number}")
    print(
        "Sentence character span:",
        failure["sentence_start"],
        "to",
        failure["sentence_end"]
    )
    print("Chunk boundary:", failure["boundary"])
    print("\nSentence:")
    print(failure["sentence"])

### Why is this a boundary failure?

Suppose a sentence occupies:

```text
character 590 → 740
```

and the fixed chunk boundary is:

```text
600
```

Then:

```text
590 < 600 < 740
```

The boundary falls **inside the sentence**.

That means the sentence cannot remain entirely in one fixed-size chunk.

This is exactly the kind of failure recursive splitting is designed to reduce.

In [ ]:
def show_boundary_context(text, boundary, window=180):
    """Display text immediately before and after a boundary."""

    start = max(0, boundary - window)
    end = min(len(text), boundary + window)

    print("Character range:", start, "to", end)
    print("\n--- BEFORE BOUNDARY ---")
    print(text[start:boundary])
    print("\n--- BOUNDARY ---")
    print("<<< SPLIT AT CHARACTER", boundary, ">>>")
    print("\n--- AFTER BOUNDARY ---")
    print(text[boundary:end])


for number, failure in enumerate(boundary_failures, start=1):
    print("=" * 90)
    print(f"Boundary failure example {number}")
    show_boundary_context(
        long_text,
        failure["boundary"],
        window=180
    )

# 14. Why does overlap help boundary failures?

Suppose the original sentence is:

```text
A B C D E F G H I J
```

and the chunk boundary cuts between:

```text
A B C D E | F G H I J
```

Without overlap, the first chunk ends at `E` and the second begins at `F`.

With overlap, we might get:

```text
Chunk 1: A B C D E F
Chunk 2:       E F G H I J
```

Now the important context around the boundary appears in both chunks.

This is why overlap can improve retrieval when an answer spans a boundary. LangChain's documentation explicitly notes that overlap helps mitigate context loss. citeturn1search0

# 15. Compare manual and recursive splitting directly

The manual splitter always uses a hard character boundary.

The recursive splitter tries to preserve larger text units first.

From the LangChain documentation, the default separator hierarchy is approximately:

```text
paragraph
   ↓
newline
   ↓
space
   ↓
individual character
```

This means the recursive splitter attempts to keep larger natural pieces together before falling back to smaller pieces. citeturn1search0

In [ ]:
comparison_size = 300
comparison_overlap = 50

manual = manual_fixed_chunks(
    long_text,
    comparison_size,
    comparison_overlap
)

recursive = RecursiveCharacterTextSplitter(
    chunk_size=comparison_size,
    chunk_overlap=comparison_overlap,
    length_function=len,
    is_separator_regex=False
).split_text(long_text)

print("Manual chunk count:", len(manual))
print("Recursive chunk count:", len(recursive))

print("\nMANUAL CHUNK #1")
print(manual[0])

print("\n" + "=" * 90)
print("RECURSIVE CHUNK #1")
print(recursive[0])

### Important distinction

**Manual fixed-size splitting** is useful for learning because it is predictable:

```text
every N characters
```

**Recursive splitting** is more context-aware:

```text
try paragraph
→ try newline
→ try space
→ finally split smaller
```

That usually makes recursive splitting a better default for ordinary prose. LangChain currently recommends it as a general-purpose splitter. citeturn1search1

# 16. Recommendation

We now turn the experiment into a practical recommendation.

A good recommendation should consider:

1. retrieval quality;
2. contextual completeness;
3. number of chunks;
4. amount of redundancy caused by overlap;
5. the structure of the source document.

For a long Wikipedia-style prose document, a **300–500 character chunk with moderate overlap** is a reasonable starting point. The exact winner should come from the experiment rather than being assumed in advance.

In [ ]:
# Produce a recommendation from the experiment.

# We first select configurations with good contextual completeness.
high_context = [
    result for result in valid_results
    if result["completeness"] >= 4
]

if high_context:
    recommended = max(
        high_context,
        key=lambda result: (
            result["top_score"],
            -result["num_chunks"]
        )
    )
else:
    recommended = best_contextual

print("RECOMMENDED CONFIGURATION")
print("=" * 60)
print("Chunk size:", recommended["chunk_size"])
print("Overlap:", recommended["overlap"])
print("Number of chunks:", recommended["num_chunks"])
print("Top retrieval similarity:", round(recommended["top_score"], 4))
print("Contextual completeness score:", recommended["completeness"])

print("\nReasoning:")
print(
    "This configuration produced a strong retrieval match while also "
    "providing enough surrounding text to make the retrieved passage "
    "more self-contained. The overlap helps protect information that "
    "falls near chunk boundaries, while avoiding the much larger "
    "redundancy produced by very high overlap."
)

## 17. Final written analysis

### Why chunk size matters

A chunk that is too small may contain only a fragment of an idea. Retrieval can become precise but context-poor.

A chunk that is too large can contain several unrelated ideas. Retrieval may return a lot of unnecessary material.

Therefore chunk size is a trade-off between:

```text
precision  ↔  context
```

### Why overlap matters

Overlap repeats text at neighbouring boundaries.

This costs storage and increases the number of chunks, but it reduces the chance that an important statement is separated exactly at a boundary.

### Why recursive splitting is useful

Fixed-size splitting ignores language structure.

Recursive splitting tries larger separators first, which helps preserve paragraphs and other natural units. This is why LangChain recommends `RecursiveCharacterTextSplitter` for generic text. citeturn1search0turn1search1

### Why there is no universal best setting

Different documents have different structures:

- legal documents may need larger context;
- FAQs may work well with smaller chunks;
- scientific papers may benefit from section-aware splitting;
- code needs language-aware separators.

Chunking should therefore be evaluated against the retrieval task instead of chosen purely by habit.

# 18. Submission checklist

- [x] Long Wikipedia article loaded as a plain Python string.
- [x] Document length checked against a five-page-scale minimum.
- [x] Fixed-size chunking manually implemented with a Python loop.
- [x] Chunk size is configurable.
- [x] Chunk overlap is configurable.
- [x] LangChain `RecursiveCharacterTextSplitter` implemented.
- [x] Chunk sizes 100, 300, 500, and 1000 tested.
- [x] Overlaps 0, 50, and 100 tested where valid.
- [x] Same retrieval query used across configurations.
- [x] Retrieval results compared across all configurations.
- [x] Contextual completeness evaluated.
- [x] Three sentence-boundary failures identified.
- [x] Exact character positions reported.
- [x] Manual and recursive approaches compared.
- [x] Final chunking recommendation documented.

## Final takeaway

Chunking is not simply:

> "Cut the document into pieces."

It is a retrieval-quality decision.

The goal is to create chunks that are:

- small enough to retrieve precisely,
- large enough to preserve meaning,
- overlapping enough to protect boundary context,
- and structured enough that a retrieved chunk can stand on its own.

That is why chunk size and overlap should be treated as **experimentally tunable parameters** rather than fixed constants.